# ARYA RAG: Extraction, Cleaning and Chunking

**Branch:** `feature/rag-extraction-chunking`

This notebook turns uploaded bytes into clean, source-aware chunks ready for embedding. It covers steps 2 and 3 of the ingestion pipeline: **extract and clean**, then **split into chunks**.

**In scope:** file validation, SHA-256 hash, PDF/TXT/MD text extraction, cleaning, chunking with overlap, page and section metadata, tests.
**Out of scope:** embeddings, database or pgvector, file storage, FastAPI routes (later branches).

**Output of this branch:** `extract_and_chunk(data, filename) -> (sha256, list[TextChunk])`. The embedding branch will turn each `TextChunk` into a `ChunkRecord` from the contract.

**Important model-limit warning:** `all-MiniLM-L6-v2` truncates input at **256 tokens**. Chunks of 600 to 800 tokens would have their tail silently ignored by the embedder. So `ChunkingConfig` is configurable, and the default below (200 tokens, 30 overlap) fits MiniLM. If the team picks a model with a longer window (for example 512 tokens or more), raise the numbers. Tell the embedding owner and update the guide, which currently says 600 to 800.

## 0. Install

In [1]:
%pip install -q pypdf

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and contract fallbacks

In [2]:
from __future__ import annotations

import hashlib
import math
import re
from collections import Counter
from dataclasses import dataclass, field
from enum import Enum
from io import BytesIO
from pathlib import PurePath
from typing import Protocol

# Use the real contract if the notebook runs inside the repo, otherwise a stand-in.
try:
    from app.rag.contracts import FileType, IngestionError, IngestionErrorCode
except ImportError:
    class FileType(str, Enum):
        PDF = "pdf"; TXT = "txt"; MD = "md"

    class IngestionErrorCode(str, Enum):
        UNSUPPORTED_FILE_TYPE = "UNSUPPORTED_FILE_TYPE"
        FILE_TOO_LARGE = "FILE_TOO_LARGE"
        NO_EXTRACTABLE_TEXT = "NO_EXTRACTABLE_TEXT"
        EMBEDDING_FAILED = "EMBEDDING_FAILED"
        INDEXING_FAILED = "INDEXING_FAILED"
        DUPLICATE_DOCUMENT = "DUPLICATE_DOCUMENT"

    class IngestionError(Exception):
        def __init__(self, code, message):
            super().__init__(message)
            self.code, self.message = code, message

## 2. Configuration and token counting

In [3]:
@dataclass(frozen=True)
class ChunkingConfig:
    target_tokens: int = 200      # keep <= embedding model max tokens (MiniLM = 256)
    overlap_tokens: int = 30
    max_file_bytes: int = 10 * 1024 * 1024   # 10 MB; enforced BEFORE parsing

    def __post_init__(self):
        if self.target_tokens < 20:
            raise ValueError("target_tokens too small")
        if not 0 <= self.overlap_tokens < self.target_tokens // 2:
            raise ValueError("overlap_tokens must be >= 0 and < half of target_tokens")


class TokenCounter(Protocol):
    def count(self, text: str) -> int: ...


class ApproxTokenCounter:
    """No-dependency estimate: roughly 1.3 tokens per word.
    Swap for the embedding model's real tokenizer for exact sizes."""
    def count(self, text: str) -> int:
        return math.ceil(len(text.split()) * 1.3)


# Optional exact counter (needs `transformers`; use when the embedder is chosen):
# from transformers import AutoTokenizer
# class HFTokenCounter:
#     def __init__(self, name="sentence-transformers/all-MiniLM-L6-v2"):
#         self.tok = AutoTokenizer.from_pretrained(name)
#     def count(self, text): return len(self.tok.encode(text, add_special_tokens=False))

## 3. Validation and hashing

Checks happen **before parsing**: extension first, then size. File type comes from the extension and is then confirmed against content (PDFs must start with `%PDF`).

In [4]:
_EXT_TO_TYPE = {".pdf": FileType.PDF, ".txt": FileType.TXT, ".md": FileType.MD, ".markdown": FileType.MD}


def sha256_hex(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def detect_file_type(filename: str, data: bytes, config: ChunkingConfig) -> FileType:
    ext = PurePath(filename or "").suffix.lower()
    ftype = _EXT_TO_TYPE.get(ext)
    if ftype is None:
        raise IngestionError(IngestionErrorCode.UNSUPPORTED_FILE_TYPE,
                             "Only PDF, TXT and Markdown files are supported.")
    if len(data) > config.max_file_bytes:
        raise IngestionError(IngestionErrorCode.FILE_TOO_LARGE,
                             f"File exceeds the {config.max_file_bytes // (1024 * 1024)} MB limit.")
    if len(data) == 0:
        raise IngestionError(IngestionErrorCode.NO_EXTRACTABLE_TEXT, "The file is empty.")
    if ftype is FileType.PDF and not data.lstrip()[:5] == b"%PDF-":
        raise IngestionError(IngestionErrorCode.UNSUPPORTED_FILE_TYPE,
                             "The file is not a valid PDF.")
    return ftype

## 4. Extraction

Every format becomes a list of `ExtractedPage`. TXT and MD have a single page with `page=None` because they have no real pages. PDFs use 1-based page numbers.

In [5]:
@dataclass
class ExtractedPage:
    page: int | None
    text: str


def extract_pages(data: bytes, file_type: FileType) -> list[ExtractedPage]:
    if file_type is FileType.PDF:
        return _extract_pdf(data)
    return [ExtractedPage(page=None, text=_decode_text(data))]


def _decode_text(data: bytes) -> str:
    for enc in ("utf-8-sig", "utf-16"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            continue
    return data.decode("latin-1", errors="replace")


def _extract_pdf(data: bytes) -> list[ExtractedPage]:
    try:
        from pypdf import PdfReader
        reader = PdfReader(BytesIO(data))
        if reader.is_encrypted:
            raise IngestionError(IngestionErrorCode.UNSUPPORTED_FILE_TYPE,
                                 "Password-protected PDFs are not supported.")
        pages = [ExtractedPage(page=i, text=(p.extract_text() or ""))
                 for i, p in enumerate(reader.pages, start=1)]
    except IngestionError:
        raise
    except Exception:
        # Do not leak parser internals or file content to the client
        raise IngestionError(IngestionErrorCode.UNSUPPORTED_FILE_TYPE,
                             "The PDF could not be read.")
    return pages

## 5. Cleaning

- Normalise line endings, tabs, odd Unicode spaces, and remove null bytes.
- Re-join words hyphenated across line breaks (`classi-\nfication`).
- Remove page-number-only lines (`4`, `Page 4 of 20`).
- Remove short lines repeated across many pages (running headers and footers).
- For PDF and TXT, join hard-wrapped lines inside a paragraph. Markdown keeps its line structure.

Raises `NO_EXTRACTABLE_TEXT` when nothing is left (for example a scanned PDF).

In [6]:
_PAGE_NUM_RE = re.compile(r"^\s*(page\s+)?\d{1,4}(\s*(of|/)\s*\d{1,4})?\s*$", re.IGNORECASE)
_HYPHEN_BREAK_RE = re.compile(r"(\w)-\n(\w)")


def _basic_normalise(text: str) -> str:
    text = text.replace("\x00", "").replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ").replace("\u200b", "").replace("\t", " ")
    text = _HYPHEN_BREAK_RE.sub(r"\1\2", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = "\n".join(line.rstrip() for line in text.split("\n"))
    return text


def _edge_lines(text: str, n: int = 2) -> list[str]:
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return lines[:n] + lines[-n:]


def _signature(line: str) -> str:
    return re.sub(r"\d+", "#", line.lower())


def _find_repeated_artifacts(pages: list[ExtractedPage]) -> set[str]:
    if len(pages) < 3:
        return set()
    counts: Counter[str] = Counter()
    for p in pages:
        for sig in {_signature(l) for l in _edge_lines(p.text) if len(l) <= 80}:
            counts[sig] += 1
    return {sig for sig, c in counts.items() if c / len(pages) >= 0.5}


_NUMBERED_HEADING_RE = re.compile(r"^\d+(\.\d+)*[.)]?\s+\S")
_SMALL_WORDS = {"and", "of", "the", "in", "to", "for", "a", "an", "vs", "with", "on"}


def _is_heading_line(s: str) -> bool:
    """Heuristic for a single line that looks like a heading (PDF/TXT only)."""
    s = s.strip()
    if len(s) < 3 or len(s) > 80 or s.endswith((".", ",", ";", ":")):
        return False
    words = s.split()
    if _NUMBERED_HEADING_RE.match(s) and len(words) <= 10:
        return True
    if s.isupper() and len(words) <= 10:
        return True
    return len(words) <= 8 and all(
        w[0].isupper() or not w[0].isalpha() or w.lower() in _SMALL_WORDS for w in words)


def _join_wrapped(text: str) -> str:
    """Join hard-wrapped lines into paragraphs, but keep heading-like lines separate."""
    out: list[str] = []
    for block in re.split(r"\n\s*\n", text):
        cur: list[str] = []
        for line in (l.strip() for l in block.split("\n") if l.strip()):
            if _is_heading_line(line):
                if cur:
                    out.append(" ".join(cur))
                    cur = []
                out.append(line)
            else:
                cur.append(line)
        if cur:
            out.append(" ".join(cur))
    return "\n\n".join(out)


def clean_pages(pages: list[ExtractedPage], file_type: FileType) -> list[ExtractedPage]:
    normalised = [ExtractedPage(p.page, _basic_normalise(p.text)) for p in pages]
    repeated = _find_repeated_artifacts(normalised) if file_type is FileType.PDF else set()

    cleaned: list[ExtractedPage] = []
    for p in normalised:
        kept = []
        for line in p.text.split("\n"):
            s = line.strip()
            if s and _PAGE_NUM_RE.match(s) and file_type is FileType.PDF:
                continue
            if s and len(s) <= 80 and _signature(s) in repeated:
                continue
            kept.append(line)
        text = "\n".join(kept)
        if file_type is not FileType.MD:
            # join hard-wrapped lines inside paragraphs
            text = _join_wrapped(text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        cleaned.append(ExtractedPage(p.page, text))

    if not any(p.text for p in cleaned):
        raise IngestionError(
            IngestionErrorCode.NO_EXTRACTABLE_TEXT,
            "No extractable text found. Scanned PDFs are not supported yet.")
    return cleaned

## 6. Chunking

Strategy:
1. Split each page into paragraphs, then into sentences. These are the smallest units.
2. Track the **current section** (Markdown `#` headings, or heading-like lines in PDF/TXT) and the **page** of every unit.
3. A heading is attached to the content that follows it, so a chunk never ends on a dangling title.
4. Pack units up to `target_tokens`. The next chunk starts with the last ~`overlap_tokens` of the previous one.
5. A single sentence larger than the target is split by words so no chunk exceeds the limit.

Each chunk records the page and section of its first unit, which is what the citation will show.

In [7]:
@dataclass
class TextChunk:
    chunk_index: int
    text: str
    token_count: int
    page: int | None = None
    section: str | None = None


@dataclass
class _Unit:
    text: str
    tokens: int
    page: int | None
    section: str | None
    para_break: bool          # True if a paragraph break precedes this unit


_MD_HEADING_RE = re.compile(r"^\s{0,3}(#{1,6})\s+(.*?)\s*#*\s*$")
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(\[])")


def _heading_text(block: str, file_type: FileType) -> str | None:
    """Return the heading text if the block is a heading, else None."""
    first = block.strip().split("\n")[0]
    if file_type is FileType.MD:
        m = _MD_HEADING_RE.match(first)
        if m and "\n" not in block.strip():
            return m.group(2).strip() or None
        return None
    s = block.strip()
    return s if ("\n" not in s and _is_heading_line(s)) else None


def _split_long(text: str, limit: int, counter: TokenCounter) -> list[str]:
    """Split text that exceeds `limit` tokens into word-based pieces."""
    if counter.count(text) <= limit:
        return [text]
    words, pieces, cur = text.split(), [], []
    for w in words:
        cur.append(w)
        if counter.count(" ".join(cur)) > limit and len(cur) > 1:
            cur.pop()
            pieces.append(" ".join(cur))
            cur = [w]
    if cur:
        pieces.append(" ".join(cur))
    return pieces


def _build_units(pages: list[ExtractedPage], file_type: FileType,
                 config: ChunkingConfig, counter: TokenCounter) -> list[_Unit]:
    units: list[_Unit] = []
    section: str | None = None
    pending_heading: str | None = None

    for p in pages:
        for block in re.split(r"\n\s*\n", p.text):
            block = block.strip()
            if not block:
                continue
            heading = _heading_text(block, file_type)
            if heading:
                section = heading
                pending_heading = block if pending_heading is None else pending_heading + "\n" + block
                continue
            sentences = [s.strip() for s in _SENTENCE_SPLIT_RE.split(block) if s.strip()] \
                if file_type is not FileType.MD else [block]
            first = True
            for sent in sentences:
                if first and pending_heading:
                    sent = pending_heading + "\n" + sent
                    pending_heading = None
                for piece in _split_long(sent, config.target_tokens, counter):
                    units.append(_Unit(piece, counter.count(piece), p.page, section,
                                       para_break=first))
                    first = False

    if pending_heading:   # document ends with a heading only
        units.append(_Unit(pending_heading, counter.count(pending_heading), pages[-1].page, section, True))
    return units


def _join(units: list[_Unit]) -> str:
    out = []
    for i, u in enumerate(units):
        if i:
            out.append("\n\n" if u.para_break else " ")
        out.append(u.text)
    return "".join(out)


def chunk_pages(pages: list[ExtractedPage], file_type: FileType,
                config: ChunkingConfig | None = None,
                counter: TokenCounter | None = None) -> list[TextChunk]:
    config = config or ChunkingConfig()
    counter = counter or ApproxTokenCounter()
    units = _build_units(pages, file_type, config, counter)

    chunks: list[TextChunk] = []
    start, n = 0, len(units)
    while start < n:
        end, total = start, 0
        while end < n and (end == start or total + units[end].tokens <= config.target_tokens):
            total += units[end].tokens
            end += 1
        text = _join(units[start:end])
        chunks.append(TextChunk(len(chunks), text, counter.count(text),
                                units[start].page, units[start].section))
        if end >= n:
            break
        back, ov = end, 0                      # walk back to build the overlap
        while back - 1 > start and ov + units[back - 1].tokens <= config.overlap_tokens:
            back -= 1
            ov += units[back].tokens
        start = back                           # always > previous start, so no infinite loop
    return chunks

## 7. One entry point

`extract_and_chunk` is the only function the ingestion service (and later the API) needs. It never touches a database, the embedder, or storage.

In [8]:
def extract_and_chunk(data: bytes, filename: str,
                      config: ChunkingConfig | None = None,
                      counter: TokenCounter | None = None) -> tuple[str, FileType, list[TextChunk]]:
    config = config or ChunkingConfig()
    file_type = detect_file_type(filename, data, config)       # validates BEFORE parsing
    digest = sha256_hex(data)
    pages = clean_pages(extract_pages(data, file_type), file_type)
    chunks = chunk_pages(pages, file_type, config, counter)
    if not chunks:
        raise IngestionError(IngestionErrorCode.NO_EXTRACTABLE_TEXT, "No extractable text found.")
    return digest, file_type, chunks

## 8. Demo

In [9]:
sample_md = b'''# Support Vector Machines

A support vector machine finds a hyperplane that separates classes. The margin is the distance between the hyperplane and the closest points. Those closest points are called support vectors.

## Kernels

Kernels let an SVM handle nonlinear data. The kernel trick maps inputs into a higher dimensional space without computing the mapping explicitly. Common kernels are linear, polynomial and RBF.
'''
sample_md = (sample_md + b"\n") * 3

digest, ftype, chunks = extract_and_chunk(sample_md, "svm.md", ChunkingConfig(target_tokens=60, overlap_tokens=10))
print(digest[:12], ftype.value, len(chunks), "chunks")
for c in chunks[:3]:
    print(f"\n[{c.chunk_index}] page={c.page} section={c.section!r} tokens={c.token_count}")
    print(c.text[:150].replace("\n", " / "), "...")

42ceb718b5e0 md 6 chunks

[0] page=None section='Support Vector Machines' tokens=43
# Support Vector Machines / A support vector machine finds a hyperplane that separates classes. The margin is the distance between the hyperplane and th ...

[1] page=None section='Kernels' tokens=41
## Kernels / Kernels let an SVM handle nonlinear data. The kernel trick maps inputs into a higher dimensional space without computing the mapping explic ...

[2] page=None section='Support Vector Machines' tokens=43
# Support Vector Machines / A support vector machine finds a hyperplane that separates classes. The margin is the distance between the hyperplane and th ...


## 9. Tests

Plain `assert` checks. They cover validation, extraction (including a generated PDF when `reportlab` is available), cleaning, chunk limits, overlap, metadata, and the scanned-PDF error.

In [10]:
def raises_code(code, fn):
    try:
        fn()
    except IngestionError as e:
        return e.code == code
    return False

cfg = ChunkingConfig(target_tokens=60, overlap_tokens=10)
counter = ApproxTokenCounter()

# --- config validation
try:
    ChunkingConfig(target_tokens=100, overlap_tokens=80); raise AssertionError("should fail")
except ValueError:
    pass

# --- validation happens before parsing
assert raises_code(IngestionErrorCode.UNSUPPORTED_FILE_TYPE, lambda: extract_and_chunk(b"x", "notes.docx"))
assert raises_code(IngestionErrorCode.UNSUPPORTED_FILE_TYPE, lambda: extract_and_chunk(b"not a pdf", "a.pdf"))
assert raises_code(IngestionErrorCode.FILE_TOO_LARGE,
                   lambda: extract_and_chunk(b"a" * 50, "a.txt", ChunkingConfig(max_file_bytes=10)))
assert raises_code(IngestionErrorCode.NO_EXTRACTABLE_TEXT, lambda: extract_and_chunk(b"", "a.txt"))
assert raises_code(IngestionErrorCode.NO_EXTRACTABLE_TEXT, lambda: extract_and_chunk(b"   \n\n  ", "a.txt"))

# --- hash is stable and content-based
d1, _, _ = extract_and_chunk(b"Hello world. This is a test.", "a.txt")
d2, _, _ = extract_and_chunk(b"Hello world. This is a test.", "renamed.txt")
assert d1 == d2 == sha256_hex(b"Hello world. This is a test.")

# --- cleaning
topics = ["gradient descent", "decision trees", "neural networks", "clustering methods", "model evaluation"]
pages = [ExtractedPage(i, f"ML Course Notes\nFirst line about {t}.\nSecond line about {t} in practice.\n"
                          f"Third line with extra detail on {t}.\nBody text number {i} is here and is useful.\n{i}\nPage {i} of 5")
         for i, t in enumerate(topics, start=1)]
cleaned = clean_pages(pages, FileType.PDF)
for p in cleaned:
    assert "ML Course Notes" not in p.text, "running header should be removed"
    assert "Page " not in p.text and "Body text" in p.text

hyph = clean_pages([ExtractedPage(None, "classi-\nfication works well")], FileType.TXT)
assert "classification" in hyph[0].text

wrapped = clean_pages([ExtractedPage(None, "line one\nline two\n\nnew paragraph")], FileType.TXT)
assert wrapped[0].text == "line one line two\n\nnew paragraph"

# --- chunking limits, overlap and metadata
chunks = chunk_pages([ExtractedPage(None, sample_md.decode())], FileType.MD, cfg, counter)
assert len(chunks) > 1
assert [c.chunk_index for c in chunks] == list(range(len(chunks)))
assert all(c.token_count <= cfg.target_tokens * 1.5 for c in chunks)   # heading attach can add a little
assert all(c.text.strip() for c in chunks)
assert chunks[0].section == "Support Vector Machines"
assert any(c.section == "Kernels" for c in chunks)

# overlap: consecutive chunks share some text when sentences are small
long_text = " ".join(f"Sentence number {i} talks about gradient descent." for i in range(60))
pc = chunk_pages([ExtractedPage(None, long_text)], FileType.TXT, cfg, counter)
shared = sum(1 for a, b in zip(pc, pc[1:]) if a.text.split(". ")[-1][:20] in b.text)
assert shared >= len(pc) - 2, "neighbouring chunks should overlap"
assert all(c.token_count <= cfg.target_tokens for c in pc)

# overlap = 0 gives no repeated sentences
no_ov = chunk_pages([ExtractedPage(None, long_text)], FileType.TXT, ChunkingConfig(60, 0), counter)
joined = " ".join(c.text for c in no_ov)
assert joined.count("Sentence number 10 ") == 1

# nothing is lost: every sentence appears in at least one chunk
all_text = " ".join(c.text for c in pc)
assert all(f"Sentence number {i} " in all_text for i in range(60))

# a giant single sentence is split by words and still respects the limit
giant = "word " * 500
gc = chunk_pages([ExtractedPage(None, giant)], FileType.TXT, cfg, counter)
assert len(gc) > 1 and all(c.token_count <= cfg.target_tokens for c in gc)

# heading-only document does not crash
assert chunk_pages([ExtractedPage(None, "# Only A Heading")], FileType.MD, cfg, counter)

# --- determinism
a = chunk_pages([ExtractedPage(None, long_text)], FileType.TXT, cfg, counter)
b = chunk_pages([ExtractedPage(None, long_text)], FileType.TXT, cfg, counter)
assert [c.text for c in a] == [c.text for c in b]

# --- PDF end to end (needs reportlab to generate a test file)
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas

    def make_pdf(pages_text, blank=False):
        buf = BytesIO()
        c = canvas.Canvas(buf, pagesize=A4)
        for i, t in enumerate(pages_text, start=1):
            if not blank:
                c.drawString(72, 800, "ML Course Notes")
                y = 760
                for line in t:
                    c.drawString(72, y, line); y -= 16
                c.drawString(300, 40, f"Page {i} of {len(pages_text)}")
            c.showPage()
        c.save()
        return buf.getvalue()

    pdf = make_pdf([
        ["Support Vector Machines", "An SVM finds a separating hyperplane.", "Support vectors define the margin."],
        ["Kernels", "Kernels handle nonlinear data.", "The RBF kernel is very common."],
        ["Regularisation", "The C parameter trades margin width for errors."],
    ])
    d, ft, cks = extract_and_chunk(pdf, "ml.pdf", ChunkingConfig(target_tokens=20, overlap_tokens=5))
    assert ft is FileType.PDF and len(d) == 64
    assert [c.page for c in cks][0] == 1 and max(c.page for c in cks) == 3
    assert all("ML Course Notes" not in c.text for c in cks)
    assert any("hyperplane" in c.text for c in cks)
    assert cks[0].section == "Support Vector Machines"
    assert any(c.section == "Kernels" and c.page == 2 for c in cks)

    scanned = make_pdf(["", "", ""], blank=True)           # pages with no text layer
    assert raises_code(IngestionErrorCode.NO_EXTRACTABLE_TEXT, lambda: extract_and_chunk(scanned, "scan.pdf"))
    print("PDF tests passed")
except ImportError:
    print("reportlab not installed: PDF end-to-end tests skipped")

print("All extraction and chunking tests passed")

reportlab not installed: PDF end-to-end tests skipped
All extraction and chunking tests passed


## 10. Move this into the repository

| Notebook cells | Target file |
|---|---|
| 2 (config, token counter) | `backend/app/rag/ingestion/config.py` |
| 3 to 5 (validation, extraction, cleaning) | `backend/app/rag/ingestion/extraction.py` and `cleaning.py` |
| 6 (chunking) | `backend/app/rag/ingestion/chunking.py` |
| 7 (entry point) | `backend/app/rag/ingestion/pipeline.py` |
| 9 (tests) | `backend/tests/unit/rag/test_extraction.py`, `test_chunking.py`, rewritten as `pytest` functions |

In the real files, delete the fallback block in cell 1 and import from `app.rag.contracts`. Add `pypdf` to the backend dependencies.

**Commits (small and separate)**
```bash
git add backend/app/rag/ingestion/config.py && git commit -m "rag: add chunking config and token counter"
git add backend/app/rag/ingestion/extraction.py && git commit -m "rag: add file validation and PDF/TXT/MD extraction"
git add backend/app/rag/ingestion/cleaning.py && git commit -m "rag: add text cleaning and header/footer removal"
git add backend/app/rag/ingestion/chunking.py && git commit -m "rag: add overlapping source-aware chunker"
git add backend/tests && git commit -m "rag: add extraction and chunking tests"
```

**Before the PR**
1. Agree the chunk size with whoever picks the embedding model (see the 256-token warning at the top).
2. Run the tests against two or three real files: a text PDF, a Markdown note, and a scanned PDF. Check by eye that chunks split at sensible places.
3. Confirm the output shape (`TextChunk`) with the person writing the embedding adapter.

**Known limits (document these, do not hide them)**
- Scanned PDFs return `NO_EXTRACTABLE_TEXT`; OCR is a later stage.
- Tables and multi-column PDFs may extract in a jumbled order.
- Heading detection for PDF and TXT is a heuristic and will sometimes miss or over-detect.
- `ApproxTokenCounter` is an estimate; use the real tokenizer once the embedding model is fixed.